# 03 - Batch LLaMEA Evolution & Multi-Problem Analysis

This notebook focuses on **running LLaMEA evolutionary search across multiple BBOB problem instances** in batch mode using `run_evolution_for_problems`, **processing returned `ProblemEvolutionResult` contracts**, **saving summaries to disk**, and **visualizing/comparing convergence across problems**.

In [ ]:
# Setup paths and imports
import sys
from pathlib import Path
# Add project root src/ to sys.path regardless of directory name or launch folder
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
src_path = str(root_dir / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import os
import plotly.graph_objects as go
from dotenv import load_dotenv

from llm.providers import build_llm
from problems.bbob import BBOBProblem
from core.runner import run_evolution_for_problems
from analysis.results import save_summary, print_experiment_summary, load_summaries


## 1. Unified Batch Experiment Configuration
Define the batch problem list, dimensionality, noise standard deviation, evaluation budget, and evolution iterations.

In [ ]:
# Batch Multi-Problem Configuration
problem_ids = [1, 2]
dim = 3
noise_std = 0.05
iterations = 5
max_evaluations = 1000
mode = "noisy"

print(f"Batch Experiment Target Problems: BBOB {problem_ids} ({dim}D)")
print(f"Configuration: mode={mode}, noise_std={noise_std}, iterations={iterations}, max_evaluations={max_evaluations}")


## 2. LLM Provider Configuration
Configure LLM provider (e.g. Gemini or Local) from environment variables or `.env` file.

In [ ]:
env_path = root_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)

llm_provider = os.getenv("LLM_PROVIDER", "local")

try:
    llm = build_llm(llm_provider)
    print("=================================================================")
    print(f"Initialized Provider: {llm_provider}")
    print(f"Model Target:         {getattr(llm, 'model', 'unknown')}")
    print("=================================================================")
except Exception as e:
    print(f"Provider setup notice: {e}")
    llm = None


## 3. Execute Batch LLaMEA Evolutionary Search
Instantiate `BBOBProblem` objects and run evolution across the target problem list. `run_evolution_for_problems` returns a list of `ProblemEvolutionResult` contracts.

In [ ]:
print(f"Starting batch evolution for problems {problem_ids}...")

problem_instances = [BBOBProblem(problem_id=pid, dim=dim) for pid in problem_ids]

results = run_evolution_for_problems(
    problems=problem_instances,
    noise_std=noise_std,
    llm=llm,
    max_evaluations=max_evaluations,
    iterations=iterations,
    verbose=True,
    log=True
)

print(f"\nCompleted batch evolution across {len(results)} problem(s)!")


## 4. Save Summaries & Display Experiment Report
Iterate through the returned `ProblemEvolutionResult` contracts to save `summary.json` files, then invoke `print_experiment_summary` to view the collected experiment results table.

In [ ]:
logs_dir = root_dir / "logs"

# Process returned result contracts and save summaries explicitly
for res in results:
    if res.error_msg is None:
        problem_dir = logs_dir / res.experiment_name
        summary_path = save_summary(
            history=res.run_history,
            problem_id=res.problem_id,
            dim=res.dim,
            output_dir=problem_dir,
            mode=res.mode
        )
        print(f"Saved summary for BBOB-{res.problem_id} -> {summary_path}")
    else:
        print(f"Evolution failed for BBOB-{res.problem_id}: {res.error_msg}")

print("\n--- Experiment Summary Table ---")
print_experiment_summary(logs_dir)


## 5. Multi-Problem Convergence Comparison
Compare convergence trajectories across all evaluated problems in the batch experiment.

In [ ]:
summaries = load_summaries(logs_dir)

if summaries:
    fig = go.Figure()
    
    for s in summaries:
        pid = s.get("problem_id")
        history = s.get("iterations", [])
        iterations_x = [h["iteration"] for h in history]
        errors_y = [h.get("final_error", float('nan')) for h in history]
        
        fig.add_trace(go.Scatter(
            x=iterations_x,
            y=errors_y,
            mode='lines+markers',
            name=f"BBOB Problem {pid}"
        ))
        
    fig.update_layout(
        title="LLaMEA Multi-Problem Convergence Comparison",
        xaxis_title="Evolution Iteration",
        yaxis_title="Final Error from Optimum",
        template="plotly_white",
        height=500,
        width=900
    )
    fig.show()
else:
    print("No logged summaries found to plot.")
